# Wireless Coverage — Part 3b: Raster-Pixel Surfaces (the pixel counterpoint)

Part 3a took the LiDAR surfaces **beyond pixels** into an H3 discrete global grid. This
notebook is the honest counterpoint: it holds the native **1 m raster** representation up
next to that H3 grid, and then re-runs Part 4's tower-siting question on the **raster** side
so you can see the two paradigms score the *same* ground.

- **Part A — representation contrast.** The same Golden Gate Park window as a 1 m-pixel DSM
  raster *and* as nb3a's H3 cells. Real counts: millions of pixels vs a few hundred cells at
  res 10 — and the resolution is a *dial*, not a ceiling (a res-15 cell is sub-metre, finer
  than a 1 m pixel). Where each representation wins.
- **Part B — viewshed foil.** The strongest nb4 candidate towers, scored with the **raster**
  per-tower viewshed (`rst_viewshed_towers`) and compared to nb4's **H3-cell** viewshed on a
  common metric (coverage area within the 2.5 km disk). Raster is pixel-exact per tower; H3
  scales to a cheap screening pass that rules most sites out first.

> **Runtime.** Databricks Serverless environment 6 — the [lightweight tier](https://databrickslabs.github.io/geobrix/docs/api/execution-tiers) (pure Python/PySpark + `h3`, no JAR). `DEMO=True` runs the Golden Gate Park sub-AOI; at full scale the per-tower raster viewshed is a one-tower-per-task (Lakeflow) fan-out.

---
**Last Update:** 2026-10-05

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
%run ./config_nb

In [ ]:
# ── nb3b overrides (after %run ./config_nb) ─────────────────────────────────
# config_nb provides: F, DBF, rx, _tbl, _persist, _points_as_gdf, cells_as_gdf,
#   plot_static, plot_raster, raster_layer, vector_layer, lonlat_to_3857,
#   X0, Y0, TILE_M, TPX, PIXEL_M, CATALOG, SCHEMA, OUT_DIR.
import os, re, math, time
import h3
import geopandas as _gpd
import pyproj
from shapely.geometry import box as _box, Point as _Pt

from databricks.labs.gbx.pyrx import rst_viewshed_towers       # Task 2 — per-tower raster viewshed
from databricks.labs.gbx.ds._listing import list_files, to_local_path

DEMO            = True
FORCE_RELOAD    = False
OUT_PREFIX      = "wc_rpx"           # raster-pixel outputs: wc_rpx_*

# Siting constants mirror nb4 so Part B compares like-for-like.
RADIUS_M        = 2500.0             # 5G urban coverage buffer (nb4 MAX_DIST_M)
ANTENNA_ABOVE_SURFACE_M = 10.0       # observer this far above the DSM (nb4 antenna clearance)
TARGET_H        = 1.6                # receiver height above the surface at the target
FINE_RES        = 12                 # nb4 detailed viewshed resolution (its viewshed_cells)
TOWER_RES       = 9                  # nb4 candidate lattice (wc_h3_candidate_sites_demo_res9)
N_DEMO_TOWERS   = 5                  # a handful of the strongest nb4 sites

GGP_LON, GGP_LAT = -122.486, 37.769  # Golden Gate Park center (series demo anchor)
WIN_TILES_A     = 2                  # Part A representation window: 2x2 tiles around GGP

VRT_DIR         = f"{OUT_DIR}/_nb3b_rvs_vrt"   # durable per-tower VRTs (Volume)
_to4326         = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

# res-cell areas (equal-area on the sphere) — the resolution dial, in m^2.
A10 = h3.average_hexagon_area(10, unit="m^2")
A12 = h3.average_hexagon_area(FINE_RES, unit="m^2")
A15 = h3.average_hexagon_area(15, unit="m^2")
print(f"DEMO={DEMO} | radius={RADIUS_M:.0f}m | antenna +{ANTENNA_ABOVE_SURFACE_M:.0f}m above DSM")
print(f"H3 cell area (m^2): res 10 ~ {A10:,.0f} | res 12 ~ {A12:,.1f} | res 15 ~ {A15:,.2f}")

## Part A — the same ground, two representations

A raster is a **regular grid of square pixels**; an H3 grid is a **discrete global grid
system (DGGS)** of hexagonal cells. Over one 2x2-tile (~2 km) window around Golden Gate Park
we count how many *pixels* vs how many *cells* it takes to tile the same ground.

> **Resolution is a dial, not a ceiling.** The raster is fixed at 1 m. H3 rolls to *any*
> resolution via `h3_toparent` + a GROUP BY: a res-10 cell is ~15,000 m^2, res 12 ~307 m^2,
> and a **res-15 cell is ~0.9 m^2 — finer than a 1 m pixel** (nb1 already stored `cellid_r15`
> on every point). "Beyond pixels" is the DGGS paradigm — hierarchical, equal-area, globally
> addressable cells — not a coarser grid.

In [ ]:
# Window = 2x2 tiles around the Golden Gate Park center tile.
_gx, _gy = lonlat_to_3857(GGP_LON, GGP_LAT)
_tx0 = int((_gx - X0) // TILE_M)
_ty0 = int((_gy - Y0) // TILE_M)
_tx1, _ty1 = _tx0 + WIN_TILES_A, _ty0 + WIN_TILES_A

dsm_r = spark.table(_tbl("wc_surface", "dsm"))          # 1 m DSM STRUCT tiles (EPSG:3857)
_win = F.col("tx").between(_tx0, _tx1 - 1) & F.col("ty").between(_ty0, _ty1 - 1)
n_tiles = dsm_r.where(_win).count()
n_px = n_tiles * int(TPX) * int(TPX)                    # 1 m^2 per pixel

# Window box in lon/lat (EPSG:4326) + projected bounds (EPSG:3857) for H3 + framing.
_wx0, _wy0 = X0 + _tx0 * TILE_M, Y0 + _ty0 * TILE_M
_wx1, _wy1 = X0 + _tx1 * TILE_M, Y0 + _ty1 * TILE_M
_w, _s = _to4326.transform(_wx0, _wy0)
_e, _n = _to4326.transform(_wx1, _wy1)
_win_wkb = _box(_w, _s, _e, _n).wkb
_win_area = (WIN_TILES_A * TILE_M) ** 2                 # projected window area (m^2)


def _h3_tiling_count(res):
    """How many H3 cells cover the window box at this resolution."""
    return (spark.createDataFrame([(1,)], "g int")
            .select(F.explode(DBF.h3_try_coverash3(F.lit(_win_wkb), F.lit(res))).alias("c"))
            .where(F.col("c").isNotNull()).count())


n_h3_10 = _h3_tiling_count(10)
n_h3_12 = _h3_tiling_count(FINE_RES)
n_h3_15_est = int(_win_area / A15)                      # res 15 is ~millions — estimate, don't materialize

dsm_h3 = spark.table(_tbl("wc_h3", "dsm", 10))          # nb3a's gridded DSM surface (res 10)
n_dsm_cells_r10 = dsm_h3.select("cellid").distinct().count()

print(f"Window: {n_tiles} tile(s) tx[{_tx0}..{_tx1 - 1}] ty[{_ty0}..{_ty1 - 1}] "
      f"= {WIN_TILES_A * int(TILE_M)} m per side")
print(f"  raster 1 m pixels            : {n_px:,}")
print(f"  H3 cells @res 10 (~{A10:,.0f} m^2) : {n_h3_10:,}")
print(f"  H3 cells @res 12 (~{A12:,.0f} m^2)   : {n_h3_12:,}")
print(f"  H3 cells @res 15 (~{A15:.2f} m^2)      : ~{n_h3_15_est:,}  (sub-metre, finer than a 1 m pixel)")
print(f"nb3a stored DSM surface (res 10, demo): {n_dsm_cells_r10:,} distinct cells")

In [ ]:
# Side-by-side: the 1 m DSM raster window vs the nb3a H3-gridded DSM surface (res 10).
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: 1 m-pixel DSM raster mosaic over the window.
_ras = []
for _tx in range(_tx0, _tx1):
    for _ty in range(_ty0, _ty1):
        _row = dsm_r.where((F.col("tx") == _tx) & (F.col("ty") == _ty)).first()
        if _row is not None:
            _ras.append(raster_layer(_row["dsm"], band=1, cmap="terrain", opacity=0.9,
                                     label="DSM 1 m pixel (m)" if not _ras else None))
if _ras:
    plot_static(_ras, title=f"Raster: 1 m-pixel DSM ({n_px:,} pixels)", ax=axes[0])
    axes[0].set_xlim(_wx0, _wx1)
    axes[0].set_ylim(_wy0, _wy1)

# Right: nb3a's H3-gridded DSM surface (res 10), dissolved by elevation band, same window.
_dsm_gdf = cells_as_gdf(dsm_h3, "cellid", extra_cols=["band_level"],
                        dissolve_by="band_level", dissolve_engine="product")
_dsm_win = _dsm_gdf.to_crs(3857).cx[_wx0:_wx1, _wy0:_wy1]
plot_static(_dsm_win, column="band_level", cmap="terrain", basemap=False, ax=axes[1],
            title=f"H3 DGGS: res 10 cells (~{n_h3_10:,} cover the window)")
axes[1].set_xlim(_wx0, _wx1)
axes[1].set_ylim(_wy0, _wy1)
plt.tight_layout()

**The trade-off.**

- **Raster (regular square grid).** Dense and exact at its fixed resolution — millions of 1 m
  pixels here. But every source lives in its *own* grid and CRS, so joining DSM to CHM to a
  different sensor means **resample / warp** to a common grid first, and the pixel count
  explodes with area. (Even "1 m" is slippery: these pixels are 1 m in EPSG:3857, which is
  Mercator-inflated at this latitude — real ground pitch is sub-metre.)
- **H3 (DGGS hexagons).** A few hundred cells cover the same window at res 10. Cells are
  **globally addressable** (the same id anywhere on Earth), **equal-area**, and **hierarchical**
  — `h3_toparent` rolls any surface up or down a resolution with a GROUP BY, so DEM/DSM/CHM and
  the tower analysis all share one id space and join directly (no warp). And it is not a
  resolution ceiling: dial to res 15 and a cell is finer than the 1 m pixel.

## Part B — viewshed foil: raster pixels vs H3 cells, same towers

Part 4 sited towers with an H3-cell line-of-sight (`h3_los_visible`) over the gridded surface.
Here we score the **same** candidate towers with the **raster** per-tower viewshed
(`rst_viewshed_towers`, Task 2): for each tower it selects the in-radius 1 m DSM tiles, mints
one bytes-free VRT, runs a single pixel-level GDAL/xrspatial viewshed, and histograms the
in-radius coverage. The observer sits +10 m above the DSM (nb4's antenna clearance), the
target +1.6 m, radius 2.5 km — identical siting assumptions, a pixel-exact answer.

We take the strongest nb4 sites **whose DSM window is fully in the COG dir**: the raster viewshed
needs real surface under and around the observer, whereas nb4 binned a wider surface window — so a
few of its highest-scoring H3 sites sit over the 1 m DSM's edge and have no raster footprint.

In [ ]:
# DSM COG manifest (path, tx, ty): list the COG dir and parse dsm_<tx>_<ty>.tif basenames.
# The DSM STRUCT table has NULL .path, so path windowing MUST read the COG dir, not the table.
# The regex matches the basename (not a path-anchored dsm_... form). _tiles is the covered set.
DSM_DIR = f"{OUT_DIR}/dsm"
_tile_re = re.compile(r"dsm_(-?\d+)_(-?\d+)\.tif$")
_rows = []
for _p in list_files(DSM_DIR, filter_regex=r".*\.tif$", recursive=False):
    _m = _tile_re.search(os.path.basename(_p))
    if _m:
        _rows.append((to_local_path(_p), int(_m.group(1)), int(_m.group(2))))
assert _rows, f"no DSM COGs under {DSM_DIR}"
dsm_tiles_df = spark.createDataFrame(_rows, "path string, tx int, ty int")
_tiles = {(r[1], r[2]) for r in _rows}                  # covered (tx, ty) set
print(f"DSM COGs: {len(_rows)} tile(s) "
      f"tx[{min(r[1] for r in _rows)}..{max(r[1] for r in _rows)}] "
      f"ty[{min(r[2] for r in _rows)}..{max(r[2] for r in _rows)}]")

In [ ]:
# Reuse the strongest nb4 candidate sites (wc_h3_candidate_sites) as the demo towers, keeping
# only those whose DSM near-window is fully present in the COG dir (observer tile + all 8
# neighbours) so the raster viewshed scores real surface. The site's res-12 cell id is the tower
# id — it already is h3.str_to_int(h3.latlng_to_cell(lat, lon, 12)).
sites_tbl = _tbl("wc_h3", "candidate_sites", TOWER_RES)
assert spark.catalog.tableExists(sites_tbl), (
    f"{sites_tbl} missing — run 04_tower_viewsheds (DEMO={DEMO}) first to build the H3 sites.")


def _well_covered(x, y):
    """True when the observer tile and all 8 neighbours are present in the DSM COG dir."""
    tx = int((x - X0) // TILE_M)
    ty = int((y - Y0) // TILE_M)
    return all((tx + dx, ty + dy) in _tiles for dx in (-1, 0, 1) for dy in (-1, 0, 1))


_cand = (spark.table(sites_tbl)
         .select("cellid", "lon", "lat", "viewshed_cells", "viewshed_pct")
         .orderBy(F.col("viewshed_cells").desc()).collect())
demo, _trows = [], []
for r in _cand:
    x, y = lonlat_to_3857(float(r["lon"]), float(r["lat"]))
    if _well_covered(x, y):
        demo.append(r)
        _trows.append((int(r["cellid"]), float(x), float(y)))
    if len(demo) >= N_DEMO_TOWERS:
        break
assert demo, "no candidate site has a fully-covered DSM window"
towers_df = spark.createDataFrame(_trows, "tower_cellid long, x double, y double")

print(f"demo towers (top {len(demo)} DSM-covered nb4 sites by H3 viewshed_cells):")
for r in demo:
    print(f"  cell {r['cellid']}  ({r['lon']:.5f}, {r['lat']:.5f})  "
          f"H3 viewshed_cells={r['viewshed_cells']:,} ({r['viewshed_pct']}%)")

In [ ]:
# Per-tower raster viewshed: one bytes-free VRT window per tower, one pixel-level viewshed,
# histogrammed to in-radius coverage, reprojected to 4326. num_partitions defaults to the tower
# count — one tower per task, the natural Lakeflow fan-out at full scale.
os.makedirs(VRT_DIR, exist_ok=True)
_rvs = rst_viewshed_towers(
    towers_df, dsm_tiles_df,
    radius_m=RADIUS_M, observer_height=ANTENNA_ABOVE_SURFACE_M,
    x0=X0, y0=Y0, tile_m=TILE_M,
    target_height=TARGET_H, to_crs=4326, vrt_dir=VRT_DIR,
    tower_id_col="tower_cellid", x_col="x", y_col="y",
)
rvs = _persist(_rvs, "raster_viewshed")                 # -> wc_rpx_raster_viewshed_demo
print(f"raster viewshed -> {_tbl(OUT_PREFIX, 'raster_viewshed')}")
rvs.orderBy(F.col("visible_px").desc()).show(truncate=False)
# A tower can legitimately return 0 visible px — an observer whose DSM window is edge-clipped or
# sits in a local pit — so this is a note, not a failure. The DSM-covered tower filter above keeps
# it rare; widen the COG dir or pick more-interior sites if you see it on your own data.
_n_total = rvs.count()
_n_zero = rvs.where(F.col("visible_px") == 0).count()
if _n_zero:
    print(f"note: {_n_zero} of {_n_total} tower(s) returned 0 visible px (edge/coverage effect, "
          f"not a failure); {_n_total - _n_zero} have visible_px > 0.")
else:
    print(f"all {_n_total} towers returned visible_px > 0")

In [ ]:
# Comparable metric: coverage AREA within the 2.5 km disk. Raster = visible_px x 1 m^2
# (EPSG:3857 projected); H3 = viewshed_cells x res-12 cell area (equal-area on the sphere).
# These are not identical measurement systems: the raster area is projected m^2 (Mercator-inflated
# ~1/cos^2(lat) at this latitude) while H3 cells are true spherical area, so the disk-normalised %
# is a close comparison, not an exact one — and that projection mismatch is itself the "regular
# grid needs resample/warp to join" cost.
_h3map = spark.createDataFrame(
    [(int(r["cellid"]), int(r["viewshed_cells"]), float(r["lon"]), float(r["lat"])) for r in demo],
    "tower_cellid long, h3_viewshed_cells long, lon double, lat double")
_DISK = math.pi * RADIUS_M * RADIUS_M
compare = (rvs.join(_h3map, "tower_cellid")
           .withColumn("raster_area_m2", F.col("visible_px") * F.lit(PIXEL_M * PIXEL_M))
           .withColumn("h3_area_m2", F.col("h3_viewshed_cells") * F.lit(A12))
           .withColumn("raster_pct_disk", F.round(100.0 * F.col("raster_area_m2") / F.lit(_DISK), 1))
           .withColumn("h3_pct_disk", F.round(100.0 * F.col("h3_area_m2") / F.lit(_DISK), 1))
           .select("tower_cellid", "lon", "lat", "visible_px", "raster_area_m2", "raster_pct_disk",
                   "h3_viewshed_cells", "h3_area_m2", "h3_pct_disk")
           .orderBy(F.col("visible_px").desc()))
compare = _persist(compare, "comparison")               # -> wc_rpx_comparison_demo
print(f"raster-vs-H3 coverage -> {_tbl(OUT_PREFIX, 'comparison')}")
compare.show(truncate=False)

**The trade-off — and why the two coverage columns diverge so much.**

- **Raster viewshed — pixel-exact, per tower.** Every 1 m cell in the DSM window is tested, so
  the coverage footprint is as precise as the input. The cost is per-tower work over a dense
  grid; the pipeline keeps it Serverless-safe by minting one window VRT per tower (no raster
  bytes cross a shuffle) and fanning out **one tower per task** — the natural unit for a
  Lakeflow run at full San-Francisco scale.
- **H3 viewshed — scalable screening.** nb4 bins the surface to one **max height per ~17 m
  res-12 cell**, runs a cheap **coarse quick-pass that rules most candidate sites out** before
  any detailed viewshed, and expresses coverage as a hierarchical cell count that joins straight
  to demand, land-cover, and other H3 layers.

**Read the gap, not just the numbers.** The two columns differ by ~100–740x here (e.g. tower
…962174975: **12 raster px / 0.0% of the disk** vs **8,865 H3 cells / 14.5%**). That is the
expected signature of H3's surface binning, **not a defect**. Collapsing the 1 m surface to one
max-height value per ~17 m cell smooths out the fine terrain and structure blockers that a 1 m
raster line-of-sight respects — so a cell counts as "seen" wherever its single binned height is
visible, inflating coverage. The H3 figure is therefore an **optimistic screening estimate**; the
raster figure is the **strict, pixel-level confirmation**. Same towers, same 2.5 km disk, same
+10 m antenna: screen the lattice cheaply with H3, then confirm a shortlisted site's exact
footprint with the raster viewshed.

## Maps — a raster-pixel viewshed beside the H3-cell coverage

The strongest demo tower's **raster** viewshed (visible vs not, at 1 m) next to nb4's
**H3-cell** coverage depth (how many sited towers see each res-12 cell). Two paradigms, one
scene.

In [ ]:
# Raster-pixel viewshed map — the strongest demo tower's exact pixel viewshed. Re-reads the
# per-tower VRT that rst_viewshed_towers minted (durable on the Volume) and runs one
# rst_viewshed, rendering visible (255) vs not (0) over the DSM window.
_top = compare.orderBy(F.col("visible_px").desc()).first()
_tcell = int(_top["tower_cellid"])
_tx_m, _ty_m = lonlat_to_3857(float(_top["lon"]), float(_top["lat"]))
_vrt = f"{VRT_DIR}/tower_{_tcell}.vrt"
if os.path.exists(_vrt):
    _obs = f"POINT ({_tx_m:.6f} {_ty_m:.6f})"
    _vs = (spark.createDataFrame([(_vrt,)], "vrt string")
           .withColumn("dem", rx.rst_fromfile("vrt", materialize=False))
           .withColumn("vs", rx.rst_viewshed("dem", F.lit(_obs), ANTENNA_ABOVE_SURFACE_M,
                                             TARGET_H, RADIUS_M, materialize=True))
           .select("vs").first()["vs"])
    _twr = _gpd.GeoDataFrame(geometry=[_Pt(float(_top["lon"]), float(_top["lat"]))], crs="EPSG:4326")
    plot_static([
        raster_layer(_vs, band=1, cmap="magma", opacity=0.8,
                     label="raster viewshed (255 visible / 0 not)"),
        vector_layer(_twr, geom_col="geometry", color="#111111", opacity=0.95, label="tower"),
    ], title=f"Raster-pixel viewshed — tower {_tcell} ({int(_top['visible_px']):,} visible px)")
else:
    print(f"VRT {_vrt} not found — raster viewshed map skipped")

In [ ]:
# H3-cell viewshed map — nb4's per-cell coverage depth (how many sited towers see each res-12
# cell), the H3 paradigm's output, with the demo towers overlaid.
cov = spark.table(_tbl("wc_h3", "coverage", FINE_RES))  # wc_h3_coverage_demo_res12
_cov_gdf = cells_as_gdf(cov.select("cellid", "tower_los_count"), "cellid",
                        extra_cols=("tower_los_count",), max_rows=200_000)
_twrs = _gpd.GeoDataFrame(
    geometry=[_Pt(float(r["lon"]), float(r["lat"])) for r in demo], crs="EPSG:4326")
plot_static([
    vector_layer(_cov_gdf, geom_col="geometry", column="tower_los_count", cmap="YlOrRd",
                 opacity=0.7, label=f"towers with LOS (res {FINE_RES})"),
    vector_layer(_twrs, geom_col="geometry", color="#111111", opacity=0.95, label="demo tower"),
], title=f"H3-cell coverage depth (res {FINE_RES}) — nb4 line-of-sight over cells")

## Summary

Side by side, the raster and the H3 grid describe the *same* Golden Gate Park surface and the
*same* tower viewsheds — but they are not rivals at a fixed resolution. The raster is a dense,
exact, single-CRS grid; H3 is a hierarchical, equal-area, globally-addressable grid that dials
to any resolution (res 15 is finer than a 1 m pixel) and joins directly across surfaces,
demand, and land cover.

**Beyond pixels is DGGS superpowers at any resolution — not a resolution ceiling.** Reach for
the raster viewshed when you need a shortlisted tower's exact footprint; reach for H3 to screen
the whole lattice cheaply and to compose coverage into the rest of the analysis.

Outputs: `wc_rpx_raster_viewshed` (per-tower raster coverage) and `wc_rpx_comparison`
(raster vs H3 coverage area on the shared 2.5 km-disk metric).